# Code for the UTF-8 encoding

In [ ]:
test_string = "Hello, World!"
utf_coded = test_string.encode('utf-8')
#for accessing the bytestream, use the list() function to convert the bytes object into a list of integers
list_of_bytes = list(utf_coded)
print(utf_coded)
print(list_of_bytes)

# Might want to look into this faster method of bytestream access later

In [ ]:
mv = memoryview(utf_coded)

print(mv[0])   # 72

# The encode method returns a bytestream internally, it just needs to be explicitly passed through the bytestream printer to be shown correctly

In [ ]:
for b in utf_coded:
    print(format(b, '08b'))

# 1. Our first step is to create the vocabulary
I shall make this a dictionary, with the key being the bytestream and the value being the integral tokenID, and this shall be stored in another dictionary.

The above is SCRAPPED, too complicated to implement a dictionary of dictionaries, we shall instead use a flat dictionary, using the keys as the binary bytestreams and the values as the integral tokenIDs.

In [ ]:
vocabulary = {bytes([i]): i for i in range(256)}

In [ ]:
vocabulary["<|endoftext|>"]=len(vocabulary)+1

In [ ]:
vocabulary

We have thus achieved the creation of vocabulary with the size of 256 and the special endoftext token

# 2. Pretokenization

In [3]:
corpus = """low low low low low
lower lower widest widest widest
newest newest newest newest newest newest"""

In [ ]:
print (corpus)

In [ ]:
pretokens = corpus.split()

In [ ]:
pretokens

In [ ]:
for token in pretokens: #pretokens is a list of (possibly repeating) words which were split from a sentence by whitespace
    if token has not been encountered before:
        obtain the token.encode("utf-8") and then split it byte by byte and then store individual bytes in  a tuple of bytes objects
        the tuple of the the bytes objects is the key of the dict[tuple[bytes, ...], int]

        the int value is set to 1
    else if token has been encountered before:
        access the dict by the key and increment the int value by 1


In [ ]:
freq = {}
for token in pretokens:
    # convert to utf8
    token_bytes = token.encode("utf-8")
    # split into individual byte objects
    byte_tuple = tuple(bytes([b]) for b in token_bytes) #something importatnt to note here is that token_bytes when iterated over shall return a single byte not the bytestream for a given character :)
    # use as dictionary key
    if byte_tuple not in freq:
        freq[byte_tuple] = 1
    else:
        freq[byte_tuple] += 1

print(freq)

# 3. Frequency tabulization for byte-pairs

In [ ]:
#make a new dictionary, where the  keys are bytes-pairs and the values are the number of occurences
for i and i+1th element of byte_tuple in all possible byte_tuple keys of the freq dictionary:
    if not encountered yet: 
        occurence [key made with the combined byteseq (so two bytes long) of i and i+1] +=freq[byte_tuple]

    if encountered:
        then increase the value of the occurence by the freq[byte_tuple]

In [ ]:
from collections import defaultdict

pair_freq = defaultdict(int) 

for byte_tuple, count in freq.items():  
    for i in range(len(byte_tuple) - 1):
        # create a byte-pair 
        byte_pair = (byte_tuple[i], byte_tuple[i+1])
        pair_freq[byte_pair] += count #because count contains how many times the pretoken has occurred

pair_freq = dict(pair_freq)

print(pair_freq)

In [ ]:
argmax the pair_freq dictionary and then handle there being two elements having the same value case (use the lexicographically larger one)
now, once we have the candidate for the argmax, we will perform the "merge" operation on the bytes-pair: this merge operation goes to the freq{} dictionary and finds out which byte_tuples contain this byte_pair, and then it deletes the two byte values from the byte_tuple key and replaces it with a new bytes object containing the appended form of the bytes instead of the two diff ones

## 3.1 Lexicographically larger highest freq target

In [ ]:
max_freq = max(pair_freq.values())
candidates = [pair for pair, freq_val in pair_freq.items() if freq_val == max_freq]


max_pair = max(candidates) 

print("Chosen pair to merge:", max_pair) #we will later insert this max_pair into the vocabulary

In [ ]:
new_freq = {}  # updated freq dictionary

for byte_tuple, count in freq.items():
    i = 0
    new_tuple = []

    while i < len(byte_tuple):
        # if current and next byte form the max_pair
        if i < len(byte_tuple) - 1 and (byte_tuple[i], byte_tuple[i+1]) == max_pair:
            # merge into a single bytes object
            merged_byte = byte_tuple[i] + byte_tuple[i+1]  # append bytes objects
            new_tuple.append(merged_byte)

            if merged_byte not in vocabulary:
                vocabulary[merged_byte] = len(vocabulary)
            i += 2  # skip the next byte because we merged
        else:
            # keep the byte as is
            new_tuple.append(byte_tuple[i])
            i += 1

    new_freq[tuple(new_tuple)] = count

freq = new_freq

print("Updated freq dictionary after merge:")
print (freq)

# 4. Final Full code

In [4]:
from collections import defaultdict

def byte_bpe(pretokens, num_iterations=5):
    freq = {}
    for token in pretokens:
        token_bytes = token.encode("utf-8")
        byte_tuple = tuple(bytes([b]) for b in token_bytes)
        freq[byte_tuple] = freq.get(byte_tuple, 0) + 1

    print("init freq dictionary:")
    print(freq)

    # bpe merger
    for iteration in range(num_iterations):
        pair_freq = defaultdict(int)
        for byte_tuple, count in freq.items():
            for i in range(len(byte_tuple) - 1):
                byte_pair = (byte_tuple[i], byte_tuple[i+1])
                pair_freq[byte_pair] += count

        if not pair_freq:
            print(f"No more pairs to merge at iteration {iteration}.")
            break

        # lexicographic max freq
        max_freq = max(pair_freq.values())
        candidates = [pair for pair, freq_val in pair_freq.items() if freq_val == max_freq]
        max_pair = max(candidates)

        print(f"Iteration {iteration+1}: Chosen pair to merge:", max_pair)

        # Merge the chosen pair in all tokens
        new_freq = {}
        for byte_tuple, count in freq.items():
            i = 0
            new_tuple = []
            while i < len(byte_tuple):
                if i < len(byte_tuple) - 1 and (byte_tuple[i], byte_tuple[i+1]) == max_pair:
                    merged_byte = byte_tuple[i] + byte_tuple[i+1]
                    new_tuple.append(merged_byte)
                    i += 2
                else:
                    new_tuple.append(byte_tuple[i])
                    i += 1
            new_freq[tuple(new_tuple)] = count
        freq = new_freq

        print(f"Updated freq dictionary after iteration {iteration+1}:")
        print(freq)

    return freq

pretokens = corpus.split()
final_freq = byte_bpe(pretokens, num_iterations=6)

init freq dictionary:
{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'e', b's', b't'): 3, (b'n', b'e', b'w', b'e', b's', b't'): 6}
Iteration 1: Chosen pair to merge: (b's', b't')
Updated freq dictionary after iteration 1:
{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'e', b'st'): 3, (b'n', b'e', b'w', b'e', b'st'): 6}
Iteration 2: Chosen pair to merge: (b'e', b'st')
Updated freq dictionary after iteration 2:
{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'n', b'e', b'w', b'est'): 6}
Iteration 3: Chosen pair to merge: (b'o', b'w')
Updated freq dictionary after iteration 3:
{(b'l', b'ow'): 5, (b'l', b'ow', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'n', b'e', b'w', b'est'): 6}
Iteration 4: Chosen pair to merge: (b'l', b'ow')
Updated freq dictionary after iteration 4:
{(b'low',): 5, (b'low', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'n', b'e', b'w', b'est'): 6}
Iterat

# 5.  Incorporating into original vocabulary

In [5]:
from collections import defaultdict
vocabulary = {bytes([i]): i for i in range(256)}
vocabulary["<|endoftext|>"]=len(vocabulary)+1
corpus = """low low low low low
lower lower widest widest widest
newest newest newest newest newest newest"""
pretokens = corpus.split()
def byte_bpe(pretokens, num_iterations=5):
    freq = {}
    for token in pretokens:
        token_bytes = token.encode("utf-8")
        byte_tuple = tuple(bytes([b]) for b in token_bytes)
        freq[byte_tuple] = freq.get(byte_tuple, 0) + 1

    print("init freq dictionary:")
    print(freq)

    # bpe merger
    for iteration in range(num_iterations):
        pair_freq = defaultdict(int)
        for byte_tuple, count in freq.items():
            for i in range(len(byte_tuple) - 1):
                byte_pair = (byte_tuple[i], byte_tuple[i+1])
                pair_freq[byte_pair] += count

        if not pair_freq:
            print(f"No more pairs to merge at iteration {iteration}.")
            break

        # lexicographic max freq
        max_freq = max(pair_freq.values())
        candidates = [pair for pair, freq_val in pair_freq.items() if freq_val == max_freq]
        max_pair = max(candidates)

        print(f"Iteration {iteration+1}: Chosen pair to merge:", max_pair)

        # Merge the chosen pair in all tokens
        new_freq = {}
        for byte_tuple, count in freq.items():
            i = 0
            new_tuple = []
            while i < len(byte_tuple):
                if i < len(byte_tuple) - 1 and (byte_tuple[i], byte_tuple[i+1]) == max_pair:
                    merged_byte = byte_tuple[i] + byte_tuple[i+1]
                    new_tuple.append(merged_byte)
                    #added rule for adding merged_byte to vocabulary
                    if merged_byte not in vocabulary:
                        vocabulary[merged_byte] = len(vocabulary)
                    i += 2
                else:
                    new_tuple.append(byte_tuple[i])
                    i += 1
            new_freq[tuple(new_tuple)] = count
        freq = new_freq

        print(f"Updated freq dictionary after iteration {iteration+1}:")
        print(freq)

    return freq

final_freq = byte_bpe(pretokens, num_iterations=6)
print (vocabulary)

init freq dictionary:
{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'e', b's', b't'): 3, (b'n', b'e', b'w', b'e', b's', b't'): 6}
Iteration 1: Chosen pair to merge: (b's', b't')
Updated freq dictionary after iteration 1:
{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'e', b'st'): 3, (b'n', b'e', b'w', b'e', b'st'): 6}
Iteration 2: Chosen pair to merge: (b'e', b'st')
Updated freq dictionary after iteration 2:
{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'n', b'e', b'w', b'est'): 6}
Iteration 3: Chosen pair to merge: (b'o', b'w')
Updated freq dictionary after iteration 3:
{(b'l', b'ow'): 5, (b'l', b'ow', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'n', b'e', b'w', b'est'): 6}
Iteration 4: Chosen pair to merge: (b'l', b'ow')
Updated freq dictionary after iteration 4:
{(b'low',): 5, (b'low', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'n', b'e', b'w', b'est'): 6}
Iterat